In [1]:
import os
import cv2
import torch
import numpy as np

from PIL import Image, ImageEnhance
from pdf2image import convert_from_path

from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    DonutProcessor
)



PDF_PATH = r"C:\Users\pardh\Downloads\PDP\24-25 Assignment 1\Please upload your assignment file (in .pdf format) (File responses)\22BCS001 - ABHIGYAN NIRANJAN IIIT Dharwad.pdf"

POPPLER_PATH = r"C:\Users\pardh\Downloads\PDP\Release-26.02.0-0\poppler-26.02.0\Library\bin"

OUTPUT_FILE = "output_final.txt"

device = "cuda" if torch.cuda.is_available() else "cpu"



print("Loading TrOCR...")

trocr_processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-handwritten"
)

trocr_model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-handwritten"
).to(device)

print("Loading Donut...")

donut_processor = DonutProcessor.from_pretrained(
    "naver-clova-ix/donut-base"
)

donut_model = VisionEncoderDecoderModel.from_pretrained(
    "naver-clova-ix/donut-base"
).to(device)

print("Models loaded successfully")



def enhance_line(img):
    img = img.convert("L")

    img = ImageEnhance.Contrast(img).enhance(2.0)

    img = img.point(lambda x: 0 if x < 180 else 255, "1")

    return img.convert("RGB")



def segment_lines(page_img):
    img = np.array(page_img)

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    thresh = cv2.threshold(
        gray,
        180,
        255,
        cv2.THRESH_BINARY_INV
    )[1]

    kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (40, 5)
    )

    dilated = cv2.dilate(
        thresh,
        kernel,
        iterations=2
    )

    contours, _ = cv2.findContours(
        dilated,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    boxes = []

    for c in contours:
        x, y, w, h = cv2.boundingRect(c)

        if w > 80 and h > 15:
            boxes.append((x, y, w, h))

    boxes = sorted(boxes, key=lambda b: b[1])

    cropped_lines = []

    for x, y, w, h in boxes:
        crop = img[y:y+h, x:x+w]
        cropped_lines.append(Image.fromarray(crop))

    return cropped_lines



def run_trocr(img):
    pixel_values = trocr_processor(
        images=img,
        return_tensors="pt"
    ).pixel_values.to(device)

    generated_ids = trocr_model.generate(
        pixel_values,
        max_new_tokens=80
    )

    text = trocr_processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return text.strip()


def run_donut(page_img):
    prompt = "<s_docvqa><s_question>Read all text</s_question><s_answer>"

    decoder_input_ids = donut_processor.tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(device)

    pixel_values = donut_processor(
        page_img,
        return_tensors="pt"
    ).pixel_values.to(device)

    outputs = donut_model.generate(
        pixel_values,
        decoder_input_ids=decoder_input_ids,
        max_length=2048,
        early_stopping=True,
        pad_token_id=donut_processor.tokenizer.pad_token_id,
        eos_token_id=donut_processor.tokenizer.eos_token_id
    )

    text = donut_processor.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0]

    return text


def poor_text(text):
    if len(text.strip()) < 20:
        return True

    words = text.split()

    if len(words) < 5:
        return True

    return False



print("Converting PDF pages...")

pages = convert_from_path(
    PDF_PATH,
    dpi=300,
    poppler_path=POPPLER_PATH
)

print("Total pages:", len(pages))

final_text = ""

for page_num, page in enumerate(pages, 1):

    print(f"Processing page {page_num}")

    lines = segment_lines(page)

    page_text = ""

    for line in lines:

        try:
            line = enhance_line(line)

            text = run_trocr(line)

            if text:
                page_text += text + "\n"

        except:
            continue

    if poor_text(page_text):

        print("Fallback → Donut")

        try:
            page_text = run_donut(page)

        except:
            pass

    final_text += f"\n===== PAGE {page_num} =====\n"
    final_text += page_text + "\n"


with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(final_text)

print("\nDone")
print(final_text)
print(f"Saved to {OUTPUT_FILE}")

C:\Users\pardh\Downloads\PDP\pdp_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading TrOCR...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 478/478 [00:00<00:00, 5392.12it/s]
[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading Donut...


Loading weights: 100%|████████████████████████████████████████████████████████████| 484/484 [00:00<00:00, 18572.11it/s]


Models loaded successfully
Converting PDF pages...
Total pages: 7
Processing page 1
Processing page 2
Processing page 3
Processing page 4
Processing page 5
Processing page 6
Processing page 7

Done

===== PAGE 1 =====
CS304 Artificial Intelligence
Assignment 2
Name : Abhigyan Niranjan
Roll no. 22BCS001
Q1 : " As per the law , it is a crime for an American to sell weapons
to hostile nations . Country A , an enemy of America , has some
missiles , and all the missiles were sold to it by Robert , who is an
American citizen . "
Now , prove that " Robert is a criminal . "
Given :
weapon ( R )
hostile ( a ) .
0 American (p ) .
2 A
2 A
2 A
sells (p. q. r ) - criminal ( p )
" enemy (A , America )
0 missile ( 1 ) " owns (A. 1 )
0 VP missile(p ) " owns (A. p. - sells ( Robert ,
A , P.P.
0 American ( Robert )
Additionally .
0 missile(p ) - weapon (p )


===== PAGE 2 =====
" enemy (p. America ) - hostile ( p )
Proof using Forward Checking :
Step 1 .
0 0
Step 2
52 1
Step 3 .
0 1


===== PAGE 3 =====